In [1]:
import os as _layout_os
from pathlib import Path as _LayoutPath
_layout_root = next(p for p in (_LayoutPath.cwd(), *_LayoutPath.cwd().parents)
                    if (p / "src" / "benchmarks.py").is_file() and (p / "scripts").is_dir())
_layout_os.chdir(_layout_root)

import os
import re
import json
import glob
import pandas as pd
from pathlib import Path

In [2]:
# ---------- Configuration ----------
# Add / remove root folders as needed. Each holds the same nested structure:
# <root>/results_experiment/<model>/<benchmark>/<benchmark>_<modelid>_<img-scale>_<timestamp>.json
ROOTS = [
    "evaluated1/results0",       # original resolution
    'evaluated1/results_interpolation',
    'evaluated1/results_field_center_aspect',
    'evaluated1/results_field_urc_aspect',
    'evaluated1/results_field_upper_right_corner',
    'evaluated1/results_field'
]

# Optional: short label per root used in column names (exp0_<scale>, exp1_<scale>, ...)
ROOT_LABELS = {root: f"exp{i}" for i, root in enumerate(ROOTS)}

OUTPUT_DIR = "model_result_tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FNAME_RE = re.compile(
    r"^(?P<benchmark>.+?)_(?P<modelid>.+?)_(?P<scale>[^_]+).json$"
)


def find_json_files(root):
    return glob.glob(os.path.join(root, "**", "*.json"), recursive=True)


def parse_filename(fname):
    m = FNAME_RE.match(os.path.basename(fname))
    return m.groupdict() if m else None


def load_records(roots):
    rows = []
    for root in roots:
        if not os.path.isdir(root):
            print(f"[warn] missing root: {root}")
            continue
        for fpath in find_json_files(root):
            meta = parse_filename(fpath)
            if meta is None:
                print(f"[warn] cannot parse filename: {fpath}")
                continue

            parts = Path(fpath).parts
            try:
                idx = parts.index("results_experiment")
                model_dir = parts[idx + 1]
                benchmark_dir = parts[idx + 2]
            except (ValueError, IndexError):
                model_dir = meta["modelid"]
                benchmark_dir = meta["benchmark"]

            try:
                with open(fpath, "r") as f:
                    data = json.load(f)
            except json.JSONDecodeError as e:
                print(f"[warn] bad JSON {fpath}: {e}")
                continue

            if isinstance(data, dict):
                records = data.get("results", [data]) if "results" in data else [data]
            else:
                records = data

            for rec in records:
                rows.append({
                    "id":            rec.get("id"),
                    "question":      rec.get("question"),
                    "ground_truth":  rec.get("ground_truth"),
                    "score":         rec.get("score"),
                    "model":         model_dir,
                    "benchmark":     benchmark_dir,
                    "scale":         meta["scale"],
                    "experiment":    ROOT_LABELS.get(root, root),
                    "root":          root,
                    "source_file":   fpath,
                })
    return pd.DataFrame(rows)


def natural_scale_sort(scales):
    def keyf(s):
        # try numeric sort first (e.g. "224", "0.5")
        try:
            return (0, float(s))
        except ValueError:
            return (1, s)
    return sorted(scales, key=keyf)


def build_per_model_tables(long_df, roots):
    """Return {model: wide_df} with experiments stacked horizontally."""
    key_cols = ["id", "question", "ground_truth", "benchmark"]
    tables = {}

    # global scale order so the columns align across models
    all_scales = natural_scale_sort(long_df["scale"].dropna().unique())
    print("Detected scales (in order):", all_scales)

    for model, mdf in long_df.groupby("model"):
        # average duplicates if any
        agg = (mdf.groupby(key_cols + ["experiment", "scale"], as_index=False)
                  ["score"].mean())

        # combine experiment + scale into a single column name
        agg["col"] = agg["experiment"] + "_" + agg["scale"].astype(str)

        wide = agg.pivot_table(
            index=key_cols,
            columns="col",
            values="score",
            aggfunc="mean",
        ).reset_index()
        wide.columns.name = None

        # build the desired column order: exp0_<scales...>, exp1_<scales...>, ...
        ordered_cols = []
        for root in roots:
            label = ROOT_LABELS.get(root, root)
            for s in all_scales:
                col = f"{label}_{s}"
                if col in wide.columns:
                    ordered_cols.append(col)

        wide = wide[key_cols + ordered_cols]
        tables[model] = wide

    return tables, all_scales


def safe_filename(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name)


# ---------- Run ----------
long_df = load_records(ROOTS)
print(f"Loaded {len(long_df)} raw records from "
      f"{long_df['source_file'].nunique() if len(long_df) else 0} files")

tables, scales = build_per_model_tables(long_df, ROOTS)

for model, df in tables.items():
    out_path = os.path.join(OUTPUT_DIR, f"{safe_filename(model)}.csv")
    df.to_csv(out_path, index=False)
    print(f"[{model}] shape={df.shape} -> {out_path}")

# Peek at one of the tables
example_model = next(iter(tables))
print(f"\nPreview of model: {example_model}")
tables[example_model].head()

Loaded 310000 raw records from 1550 files
Detected scales (in order): ['0', '0.5', '1', '1.5', '2', '3', '4', '5', '6']
[Llama-3.2-11B-Vision-Instruct] shape=(1000, 35) -> model_result_tables/Llama-3.2-11B-Vision-Instruct.csv
[Qwen2-VL-2B-Instruct] shape=(1000, 35) -> model_result_tables/Qwen2-VL-2B-Instruct.csv
[Qwen2-VL-7B-Instruct] shape=(1000, 35) -> model_result_tables/Qwen2-VL-7B-Instruct.csv
[Qwen2.5-VL-3B-Instruct] shape=(1000, 35) -> model_result_tables/Qwen2.5-VL-3B-Instruct.csv
[Qwen2.5-VL-7B-Instruct] shape=(1000, 35) -> model_result_tables/Qwen2.5-VL-7B-Instruct.csv
[Qwen3-VL-2B-Instruct] shape=(1000, 35) -> model_result_tables/Qwen3-VL-2B-Instruct.csv
[Qwen3-VL-4B-Instruct] shape=(1000, 35) -> model_result_tables/Qwen3-VL-4B-Instruct.csv
[Qwen3-VL-8B-Instruct] shape=(1000, 35) -> model_result_tables/Qwen3-VL-8B-Instruct.csv
[llava-1.5-7b-hf] shape=(1000, 35) -> model_result_tables/llava-1.5-7b-hf.csv
[pixtral-12b] shape=(1000, 35) -> model_result_tables/pixtral-12b.csv

P

,id,question,ground_truth,benchmark,exp0_0,exp1_1.5,exp1_2,exp1_3,exp1_4,exp1_5,...,exp4_2,exp4_3,exp4_4,exp4_5,exp5_0.5,exp5_1,exp5_2,exp5_3,exp5_4,exp5_5
0,1,What is the main feature in the background of ...,b,MMStar,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,6,What is the overall theme of the image?\nOptio...,a,MMStar,1.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0
2,13,Which mood does this image convey?\nOptions: A...,b,MMStar,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,23,What is the overall mood of the image?\nOption...,a,MMStar,0.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
4,39,What is the subject of the image?\nOptions: A:...,b,MMStar,1.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
